# ANU ASTR4004 2026 - Week 4: From a Signal to a Measurement

Author: Dr Sven Buder (sven.buder@anu.edu.au)

## Building and testing your own small measurement pipeline

In this tutorial, you will analyse several **mystery measurements**.

Some contain useful signals. Some may not. Some may contain signals that are not well described by the model you first try.

Your goal is **not** simply to make a fitting routine return a number.

Your goal is to decide:

> **When does a fitted number represent a trustworthy scientific measurement?**

By the end, you should have built a small pipeline that can:

- inspect and validate data,
- estimate a signal,
- fit a model,
- quantify uncertainty,
- assess detection significance,
- inspect residuals and fit quality,
- distinguish detections from non-detections and unreliable fits,
- cope with some common problems in real measurements.

---

### Ground rule

Work through the notebook from top to bottom.

**Do not scroll to the solutions until you have made a serious attempt at each task.**

There are deliberately some measurements for which the first model you try will not be adequate.

In [ ]:
try:
    %matplotlib inline
    %config InlineBackend.figure_format='retina'
except:
    pass

import numpy as np
import matplotlib.pyplot as plt

from scipy.optimize import curve_fit
from scipy.ndimage import median_filter

In [ ]:
measurements = np.load('data/pipeline_measurements.npz', allow_pickle=True)['measurements'].item()
list(measurements)

In [ ]:
measurements['measurement_0'].keys()

In [ ]:
measurements['measurement_0']['x']

# Part I — Look before you fit

## Task 1 — Inspect all measurements

Before writing any fitting code, plot all four measurements.

For each one, think about:

- Is there an obvious signal?
- Where might it be?
- How wide does it look?
- Does the measurement look like pure noise?
- Does anything look more complicated than a single peak?

### Your Task

Write a loop that produces one figure per measurement.

Plot:

- `y` versus `x`
- uncertainty bars if you find them useful
- a horizontal line at zero

Then write down which measurement you would choose **first** for developing your fitting code, and why.

> A useful research habit is to develop a new method on a case where you can understand what should happen.

In [ ]:
# YOUR CODE HERE

## Task 2 — Choose your first test case

Pick the measurement that looks like the clearest, simplest signal.

Store its name in:

```python
test_name = "measurement_?"
```

and retrieve its arrays as:

```python
test = measurements[test_name]
x_test = test["x"]
y_test = test["y"]
yerr_test = test["yerr"]
```

### Question

Why is this a better place to start than the hardest or least obvious measurement?

In [ ]:
# YOUR CODE HERE

test_name = "measurement_?"

# Part II — Build the measurement model

## Task 3 — Write down a model

For now, assume the signal can be described by:

$$
y(x) = A \exp\left[-\frac{1}{2}\left(\frac{x-\mu}{\sigma}\right)^2\right] + c
$$

where:

- $A$ is the amplitude,
- $\mu$ is the centre,
- $\sigma$ is the width,
- $c$ is a constant background.

### Your Task

Write a Python function:

```python
def gaussian_model(x, amplitude, centre, sigma, background):
    ...
```

In [ ]:
# YOUR CODE HERE

## Task 4 — Make sensible starting guesses

A numerical optimiser needs a reasonable place to start.

Without typing in values by hand from the plot, try to estimate:

- background from the median of the data,
- amplitude from the maximum relative to the background,
- centre from the location of the maximum,
- width from a rough generic guess.

For example, think about expressions involving:

```python
np.nanmedian(...)
np.nanmax(...)
np.nanargmax(...)
```

### Your Task

Create:

```python
p0 = [amplitude_guess, centre_guess, sigma_guess, background_guess]
```

Then print your initial guesses.

In [ ]:
# YOUR CODE HERE

# Part III — Fit the easiest case

## Task 5 — Fit your test measurement

Use `scipy.optimize.curve_fit`.

Useful arguments include:

```python
curve_fit(
    gaussian_model,
    x,
    y,
    p0=p0,
    sigma=yerr,
    absolute_sigma=True,
)
```

The returned values are:

```python
popt, pcov
```

where:

- `popt` contains the best-fitting parameters,
- `pcov` is their covariance matrix.

Parameter uncertainties can be estimated as:

```python
perr = np.sqrt(np.diag(pcov))
```

### Your Tasks

1. Fit your chosen test measurement.
2. Print each parameter and uncertainty.
3. Plot the data and fitted model together.
4. Does the answer look sensible?

In [ ]:
# YOUR CODE HERE

## Task 6 — Look at the residuals

A fitted curve looking approximately right is not enough.

Define the residuals as:

$$
r_i = y_i - m_i
$$

where $m_i$ is the fitted model.

### Your Task

Make a figure showing:

1. data + best-fitting model,
2. residuals underneath or in a separate figure.

Ask yourself:

- Are the residuals centred around zero?
- Do they look random?
- Is there coherent structure?
- Are there isolated extreme points?

> A good model should leave residuals that look broadly like noise.

In [ ]:
# YOUR CODE HERE

# Part IV — Turn your experiment into a reusable routine

## Task 7 — Write a fitting function

Now turn what you have done into a reusable function:

```python
def fit_measurement(x, y, yerr):
    ...
```

It should:

1. remove non-finite values,
2. create initial guesses,
3. fit the Gaussian model,
4. return best-fitting parameters and uncertainties,
5. return the model and residuals.

You can choose the exact return format.

For example, a dictionary is convenient:

```python
return {
    "params": ...,
    "errors": ...,
    "model": ...,
    "residuals": ...,
}
```

In [ ]:
# YOUR CODE HERE

## Task 8 — Apply the same routine to all four measurements

Run your fitting routine on every mystery measurement.

For each measurement:

- plot data + fitted model,
- plot residuals,
- print the fitted amplitude and its uncertainty.

### Question

Did the optimiser return an answer for every measurement?

If so:

> Does that mean every measurement contains a real signal?

In [ ]:
# YOUR CODE HERE

# Part V — A fit is not automatically a detection

## Task 9 — Define a simple detection significance

A useful first diagnostic is:

$$
S_A = \frac{A}{\sigma_A}
$$

where:

- $A$ is the fitted amplitude,
- $\sigma_A$ is its fitted uncertainty.

### Your Task

For every measurement, calculate:

```python
amplitude_significance = amplitude / amplitude_error
```

Print the result.

Then choose a provisional threshold, for example:

```python
amplitude_significance > 3
```

to define a possible detection.

### Questions

- Which measurements would pass?
- Does the noise-like spectrum pass?
- Is a significance threshold alone enough to guarantee a good measurement?

In [ ]:
# YOUR CODE HERE

## Task 10 — Measure fit quality with chi-squared

If the uncertainties are meaningful, define:

$$
\chi^2 = \sum_i \left(\frac{y_i-m_i}{\sigma_i}\right)^2
$$

and

$$
\chi^2_{\rm reduced} = \frac{\chi^2}{N-k}
$$

where:

- $N$ is the number of valid data points,
- $k$ is the number of fitted parameters.

### Your Task

Extend your fitting routine to calculate:

- `chi2`
- `dof`
- `reduced_chi2`

Print the reduced chi-squared for each measurement.

### Important question

Do not just ask whether $\chi^2_{\rm reduced}$ is close to one.

Compare it with the **shape of the residuals**.

Can you find a case where the signal is significant, but the model is still clearly inadequate?

In [ ]:
# YOUR CODE HERE

# Part VI — Decide what the pipeline should report

## Task 11 — Create measurement classes

A useful pipeline should not simply return a fitted amplitude.

Try to distinguish at least:

```python
"detection"
"non_detection"
"bad_fit"
"bad_data"
```

A first simple set of rules might involve:

- whether enough valid pixels exist,
- amplitude significance,
- whether the fitted width is physically sensible,
- reduced chi-squared,
- obvious structure in residuals.

### Your Task

Write a function such as:

```python
def classify_fit(result):
    ...
```

You will need to decide your own thresholds.

Then classify each of the four mystery measurements.

### Reflection

The thresholds do not need to be perfect.

The important question is:

> What evidence would make you trust or reject an automated measurement?

In [ ]:
# YOUR CODE HERE

# Part VII — When the model is wrong

## Task 12 — Investigate the suspicious residuals

One of the mystery measurements should show coherent residual structure when fitted with a single Gaussian.

### Your Tasks

1. Identify it.
2. Look closely at the residual pattern.
3. Ask what kind of missing structure could produce those residuals.
4. Try fitting a model containing **two components** plus a constant background.

For example:

$$
y(x)
=
G_1(x)
+
G_2(x)
+
c
$$

### Question

Does the more complex model remove the coherent residual structure?

### Bigger question

Should you automatically fit two components to *every* spectrum?

Why or why not?

In [ ]:
# YOUR CODE HERE

# Part VIII — Real data are messier

## Task 13 — A deliberately damaged measurement

Run the next cell to create one additional measurement containing several common problems.

Do not inspect the construction code too carefully before looking at the result.

In [ ]:
# Create a damaged copy of one otherwise useful spectrum.
_base_name = list(measurements.keys())[1]
_base = measurements[_base_name]

messy_measurement = {
    "x": _base["x"].copy(),
    "y": _base["y"].copy(),
    "yerr": _base["yerr"].copy(),
}

# Missing values
messy_measurement["y"][30:34] = np.nan

# One large positive outlier
messy_measurement["y"][185] += 12

# One invalid uncertainty
messy_measurement["yerr"][80] = np.nan

# One impossible uncertainty value
messy_measurement["yerr"][120] = -1

### Your Task

First, try running your current pipeline on `messy_measurement`.

What happens?

Then improve your data validation.

At minimum, require:

```python
np.isfinite(x)
np.isfinite(y)
np.isfinite(yerr)
yerr > 0
```

There is another subtle problem: if you initialise the Gaussian centre using the **single highest data point**, one extreme outlier can send the optimiser to completely the wrong place.

Try making the **initialisation** more robust, for example by locating the peak in a mildly median-filtered copy of the data:

```python
y_smooth = median_filter(y, size=7)
```

Use the smoothed data only to obtain a starting guess. Still fit the **original measurements**.

Think carefully before automatically masking the large outlier.


### Questions

- Should one unusual data point be deleted simply because it disagrees with the model?
- How could you distinguish a bad pixel from an unexpected real signal?
- What information would you want from the instrument or reduction pipeline?

In [ ]:
# YOUR CODE HERE

# Part IX — Final pipeline challenge

## Task 14 — Build your final measurement pipeline

Write a function such as:

```python
def measure_signal(x, y, yerr):
    ...
```

that returns a structured result containing at least:

- number of valid data points,
- best-fitting parameters,
- parameter uncertainties,
- amplitude significance,
- chi-squared,
- reduced chi-squared,
- residuals,
- a quality flag / classification,
- which model was finally adopted.

### A useful final piece of logic

A pipeline does not have to give up immediately when the first model is inadequate.

Try this decision tree:

```text
fit one Gaussian
       |
       +-- insignificant signal --> non-detection
       |
       +-- acceptable fit -------> single-Gaussian detection
       |
       +-- poor chi-squared -----> try two Gaussians
                                      |
                                      +-- clearly better --> adopt two components
                                      |
                                      +-- not better -----> flag as bad / suspicious fit
```

A more complicated model will almost always reduce $\chi^2$ somewhat, so do **not** adopt two Gaussians merely because their $\chi^2$ is smaller.

For this exercise, compare the models using the Bayesian Information Criterion:

$$
\mathrm{BIC} = \chi^2 + k\ln N,
$$

where $k$ is the number of fitted parameters and $N$ the number of valid measurements.

Smaller BIC is preferred. You may require something like

$$
\mathrm{BIC}_{1G}-\mathrm{BIC}_{2G}>10
$$

before accepting the more complicated model.

Run the final pipeline on all mystery measurements and summarise the results in a compact table.

In [ ]:
# YOUR CODE HERE

# Part X — Scientific reflection

Before looking at the solutions, answer these questions in words.

### 1. Why is it useful to begin method development with the cleanest-looking measurement?

### 2. Why is a successful numerical fit not sufficient evidence for a detection?

### 3. What is the difference between:

- a non-detection,
- a bad fit,
- bad data?

### 4. Why are residual plots scientifically useful?

### 5. Which is more dangerous:

- a spectrum containing only noise,
- or a strong signal described by the wrong model?

Explain your reasoning.

### 6. What information should an automated pipeline save besides the final fitted value?

In [ ]:
# Write short answers here if you wish.

---

# STOP HERE UNTIL YOU HAVE ATTEMPTED THE EXERCISES

# Full worked solutions

The code below is one reasonable solution.

It is **not** the only correct way to construct the pipeline.

The exact quality thresholds are deliberately somewhat arbitrary: in real research, they should be motivated and validated for the scientific problem and data set.

In [ ]:
rng = np.random.default_rng(173)

x = np.linspace(0, 100, 260)

def gaussian_model(x, amplitude, centre, sigma):
    return amplitude * np.exp(-0.5 * ((x - centre) / sigma)**2)

# Create four different underlying situations.
# The truth is intentionally not shown to you yet.
_truth = [
    dict(kind="noise", components=[]),
    dict(kind="single", components=[(8.5, 48.0, 5.0)]),
    dict(kind="single", components=[(2.6, 62.0, 6.5)]),
    dict(kind="double", components=[(5.0, 41.0, 3.0), (4.0, 54.0, 3.0)]),
]

measurements_raw = []

for i, truth in enumerate(_truth):
    yerr = 0.85 * (1.0 + 0.15 * np.sin(x / 17 + i))
    y_true = np.zeros_like(x)

    for amp, centre, sigma in truth["components"]:
        y_true += gaussian_model(x, amp, centre, sigma)

    y = y_true + rng.normal(0, yerr)

    measurements_raw.append({
        "x": x.copy(),
        "y": y,
        "yerr": yerr,
    })

# Shuffle the order so "measurement 0" does not mean "easy" or "hard".
order = rng.permutation(len(measurements_raw))

measurements = {
    f"measurement_{j}": measurements_raw[i]
    for j, i in enumerate(order)
}

np.savez('data/pipeline_measurements.npz', measurements=measurements)

## Solution 1 — Inspect the measurements

In [ ]:
for name, m in measurements.items():
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.errorbar(
        m["x"], m["y"], yerr=m["yerr"],
        fmt=".", ms=3, alpha=0.8, lw=1
    )
    ax.axhline(0, lw=1)
    ax.set_xlabel("x")
    ax.set_ylabel("signal")
    ax.set_title(name)
    plt.show()

A sensible choice is the measurement with the clearest isolated peak.

The exact measurement number depends on the shuffle created in the setup cell.

The reason for starting there is methodological: if the data are easy to understand visually, it is much easier to distinguish problems in the code from genuinely difficult behaviour in the data.

## Solution 2 — Gaussian model and automatic starting guesses

In [ ]:
def gaussian_model(x, amplitude, centre, sigma, background):
    return (
        amplitude
        * np.exp(-0.5 * ((x - centre) / sigma)**2)
        + background
    )


def initial_guesses(x, y):
    background_guess = np.nanmedian(y)

    # Smooth ONLY for the starting guess.
    # The actual fit still uses the original data.
    y_smooth = median_filter(y, size=7)

    amplitude_guess = np.nanmax(y_smooth) - background_guess
    centre_guess = x[np.nanargmax(y_smooth)]
    sigma_guess = 5.0

    return [
        amplitude_guess,
        centre_guess,
        sigma_guess,
        background_guess,
    ]

## Solution 3 — Choose a clean test case automatically

For the worked solution, we will simply choose the spectrum with the largest peak-to-noise ratio as a development case.

In [ ]:
peak_scores = {}

for name, m in measurements.items():
    background = np.nanmedian(m["y"])
    peak = np.nanmax(m["y"]) - background
    typical_noise = np.nanmedian(m["yerr"])
    peak_scores[name] = peak / typical_noise

test_name = max(peak_scores, key=peak_scores.get)
test = measurements[test_name]

x_test = test["x"]
y_test = test["y"]
yerr_test = test["yerr"]

print("Chosen development case:", test_name)
print("Peak scores:", peak_scores)

## Solution 4 — Fit the clean test case

In [ ]:
p0 = initial_guesses(x_test, y_test)

popt, pcov = curve_fit(
    gaussian_model,
    x_test,
    y_test,
    p0=p0,
    sigma=yerr_test,
    absolute_sigma=True,
    maxfev=10000,
)

perr = np.sqrt(np.diag(pcov))

names = ["amplitude", "centre", "sigma", "background"]

for name, value, error in zip(names, popt, perr):
    print(f"{name:12s} = {value:8.3f} +/- {error:6.3f}")

model_test = gaussian_model(x_test, *popt)

fig, ax = plt.subplots(figsize=(8, 3))
ax.errorbar(x_test, y_test, yerr=yerr_test, fmt=".", ms=3, lw=1)
ax.plot(x_test, model_test, lw=3)
ax.set_xlabel("x")
ax.set_ylabel("signal")
ax.set_title(test_name)
plt.show()

## Solution 5 — Residuals

In [ ]:
residuals_test = y_test - model_test

fig, ax = plt.subplots(figsize=(8, 3))
ax.errorbar(x_test, y_test, yerr=yerr_test, fmt=".", ms=3, lw=1)
ax.plot(x_test, model_test, lw=3)
ax.set_ylabel("signal")
ax.set_title(test_name)
plt.show()

fig, ax = plt.subplots(figsize=(8, 2.5))
ax.axhline(0, lw=1, color="k")
ax.errorbar(x_test, residuals_test, yerr=yerr_test, fmt=".", ms=3, lw=1)
ax.set_xlabel("x")
ax.set_ylabel("residual")
plt.show()

## Solution 6 — Reusable fitting routine

In [ ]:
def fit_measurement(x, y, yerr):
    x = np.asarray(x)
    y = np.asarray(y)
    yerr = np.asarray(yerr)

    good = (
        np.isfinite(x)
        & np.isfinite(y)
        & np.isfinite(yerr)
        & (yerr > 0)
    )

    x_good = x[good]
    y_good = y[good]
    yerr_good = yerr[good]

    if len(x_good) < 10:
        return {
            "success": False,
            "n_valid": len(x_good),
            "status": "bad_data",
        }

    p0 = initial_guesses(x_good, y_good)

    lower_bounds = [0.0, np.nanmin(x_good), 0.2, -np.inf]
    upper_bounds = [np.inf, np.nanmax(x_good), 50.0, np.inf]

    try:
        popt, pcov = curve_fit(
            gaussian_model,
            x_good,
            y_good,
            p0=p0,
            sigma=yerr_good,
            absolute_sigma=True,
            bounds=(lower_bounds, upper_bounds),
            maxfev=20000,
        )
    except Exception as exc:
        return {
            "success": False,
            "n_valid": len(x_good),
            "status": "bad_fit",
            "error_message": str(exc),
        }

    perr = np.sqrt(np.diag(pcov))
    model = gaussian_model(x_good, *popt)
    residuals = y_good - model
    standardized_residuals = residuals / yerr_good

    chi2 = np.sum(standardized_residuals**2)
    n_parameters = len(popt)
    dof = len(x_good) - n_parameters
    reduced_chi2 = chi2 / dof if dof > 0 else np.nan
    bic = chi2 + n_parameters * np.log(len(x_good))

    return {
        "success": True,
        "n_valid": len(x_good),
        "x": x_good,
        "y": y_good,
        "yerr": yerr_good,
        "params": popt,
        "errors": perr,
        "model": model,
        "residuals": residuals,
        "standardized_residuals": standardized_residuals,
        "max_abs_standardized_residual": np.max(
            np.abs(standardized_residuals)
        ),
        "chi2": chi2,
        "dof": dof,
        "reduced_chi2": reduced_chi2,
        "bic": bic,
        "model_type": "single_gaussian",
    }

## Solution 7 — Apply the routine to all measurements

In [ ]:
results = {}

for name, m in measurements.items():
    result = fit_measurement(m["x"], m["y"], m["yerr"])
    results[name] = result

    print("\n", name)

    if not result["success"]:
        print(result)
        continue

    amp = result["params"][0]
    amp_err = result["errors"][0]

    print(f"Amplitude = {amp:.3f} +/- {amp_err:.3f}")

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.errorbar(
        result["x"], result["y"],
        yerr=result["yerr"],
        fmt=".", ms=3, lw=1
    )
    ax.plot(result["x"], result["model"], lw=3)
    ax.set_title(name)
    ax.set_ylabel("signal")
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 2.5))
    ax.axhline(0, lw=1, color="k")
    ax.errorbar(
        result["x"], result["residuals"],
        yerr=result["yerr"],
        fmt=".", ms=3, lw=1
    )
    ax.set_xlabel("x")
    ax.set_ylabel("residual")
    plt.show()

## Solution 8 — Detection significance and fit quality

In [ ]:
for name, result in results.items():
    if not result["success"]:
        continue

    amp = result["params"][0]
    amp_err = result["errors"][0]
    significance = amp / amp_err

    result["amplitude_significance"] = significance

    print(
        f"{name:15s}  "
        f"A/sigma_A = {significance:6.2f}   "
        f"reduced chi2 = {result['reduced_chi2']:6.2f}"
    )

The noise-only spectrum should return a fitted number because the optimiser is doing exactly what it was asked to do: it finds the best Gaussian it can.

That does **not** make the Gaussian real.

Likewise, one of the spectra should have a highly significant fitted signal but visibly structured residuals. That is a model-quality problem rather than a detection-significance problem.

## Solution 9 — A simple classification function

These thresholds are intentionally simple and should not be treated as universal.

In [ ]:
def classify_fit(result):
    if not result.get("success", False):
        return result.get("status", "bad_fit")

    amp = result["params"][0]
    amp_err = result["errors"][0]
    sigma = result["params"][2]
    redchi2 = result["reduced_chi2"]

    if not np.isfinite(amp_err) or amp_err <= 0:
        return "bad_fit"

    significance = amp / amp_err

    if significance < 2:
        return "non_detection"

    if significance < 3:
        return "upper_limit"

    if redchi2 > 1.8:
        return "bad_fit"

    return "detection"


for name, result in results.items():
    status = classify_fit(result)
    result["status"] = status

    print(
        f"{name:13s}  "
        f"{status:13s}  "
        f"significance = "
        f"{result.get('params', np.nan)[0]/result.get('errors', np.nan)[0]:5.2f}  "
        f"reduced chi2 = "
        f"{result.get('reduced_chi2', np.nan):4.2f}"
    )

## Solution 10 — Two-Gaussian model

In [ ]:
def double_gaussian_model(
    x,
    amplitude_1, centre_1, sigma_1,
    amplitude_2, centre_2, sigma_2,
    background,
):
    return (
        amplitude_1
        * np.exp(-0.5 * ((x - centre_1) / sigma_1)**2)
        +
        amplitude_2
        * np.exp(-0.5 * ((x - centre_2) / sigma_2)**2)
        +
        background
    )

In [ ]:
def fit_double_gaussian(single_result):
    if not single_result.get("success", False):
        return {"success": False, "status": "bad_fit"}

    x = single_result["x"]
    y = single_result["y"]
    yerr = single_result["yerr"]

    amp1, centre1, sigma1, background = single_result["params"]

    # Search for broad missing structure in smoothed residuals.
    # This stops one isolated bad pixel from defining component 2.
    residual_smooth = median_filter(
        single_result["residuals"],
        size=7,
    )

    centre2 = x[np.argmax(residual_smooth)]
    amp2 = max(np.max(residual_smooth), np.median(yerr))

    p0 = [
        amp1, centre1, max(sigma1, 1.0),
        amp2, centre2, 3.0,
        background,
    ]

    # Do not let the second Gaussian collapse into an arbitrarily
    # narrow spike that simply absorbs one bad pixel.
    lower_bounds = [
        0.0, x.min(), 1.0,
        0.0, x.min(), 1.0,
        -np.inf,
    ]
    upper_bounds = [
        np.inf, x.max(), 30.0,
        np.inf, x.max(), 30.0,
        np.inf,
    ]

    try:
        popt, pcov = curve_fit(
            double_gaussian_model,
            x,
            y,
            p0=p0,
            sigma=yerr,
            absolute_sigma=True,
            bounds=(lower_bounds, upper_bounds),
            maxfev=30000,
        )
    except Exception as exc:
        return {
            "success": False,
            "status": "bad_fit",
            "error_message": str(exc),
        }

    perr = np.sqrt(np.diag(pcov))
    model = double_gaussian_model(x, *popt)
    residuals = y - model
    standardized_residuals = residuals / yerr

    chi2 = np.sum(standardized_residuals**2)
    n_parameters = len(popt)
    dof = len(x) - n_parameters
    reduced_chi2 = chi2 / dof
    bic = chi2 + n_parameters * np.log(len(x))

    components = [
        {
            "amplitude": popt[0],
            "amplitude_error": perr[0],
            "centre": popt[1],
            "centre_error": perr[1],
            "sigma": popt[2],
            "sigma_error": perr[2],
        },
        {
            "amplitude": popt[3],
            "amplitude_error": perr[3],
            "centre": popt[4],
            "centre_error": perr[4],
            "sigma": popt[5],
            "sigma_error": perr[5],
        },
    ]
    components = sorted(components, key=lambda c: c["centre"])

    for component in components:
        component["amplitude_significance"] = (
            component["amplitude"] / component["amplitude_error"]
        )

    return {
        "success": True,
        "n_valid": len(x),
        "x": x,
        "y": y,
        "yerr": yerr,
        "params": popt,
        "errors": perr,
        "components": components,
        "background": popt[6],
        "background_error": perr[6],
        "model": model,
        "residuals": residuals,
        "standardized_residuals": standardized_residuals,
        "max_abs_standardized_residual": np.max(
            np.abs(standardized_residuals)
        ),
        "chi2": chi2,
        "dof": dof,
        "reduced_chi2": reduced_chi2,
        "bic": bic,
        "model_type": "double_gaussian",
    }


candidate_names = [
    name for name, result in results.items()
    if result.get("success", False)
]

suspicious_name = max(
    candidate_names,
    key=lambda name: results[name]["reduced_chi2"]
)

single_result = results[suspicious_name]
double_result = fit_double_gaussian(single_result)

print("Suspicious measurement:", suspicious_name)
print("single-Gaussian reduced chi2:", f"{single_result['reduced_chi2']:.2f}")
print("double-Gaussian reduced chi2:", f"{double_result['reduced_chi2']:.2f}")
print(
    "Delta BIC (single - double):",
    f"{single_result['bic'] - double_result['bic']:.0f}",
)

fig, ax = plt.subplots(figsize=(8, 3))
ax.errorbar(
    single_result["x"],
    single_result["y"],
    yerr=single_result["yerr"],
    fmt=".",
    ms=3,
)
ax.plot(single_result["x"], single_result["model"], label="single Gaussian")
ax.plot(double_result["x"], double_result["model"], label="double Gaussian")
ax.legend()
ax.set_title(suspicious_name)
plt.show()

fig, ax = plt.subplots(figsize=(8, 2.5))
ax.axhline(0, lw=1)
ax.errorbar(
    double_result["x"],
    double_result["residuals"],
    yerr=double_result["yerr"],
    fmt=".",
    ms=3,
)
ax.set_xlabel("x")
ax.set_ylabel("double-model residual")
plt.show()

A more complicated model can fit a complicated signal better, but that does not mean it should be used automatically.

For this exercise we compare the models using

$$
\mathrm{BIC} = \chi^2 + k\ln N.
$$

Because **smaller BIC is better**, define

```python
delta_bic = bic_single - bic_double
```

and require something like `delta_bic > 10` before accepting the extra component.

Notice also that the second component is initialised from **smoothed residual structure**. We want it to respond to coherent missing structure, not to one pathological pixel.

## Solution 11 — Handle the damaged measurement

The updated `initial_guesses` function uses a median-filtered copy of the data to locate the approximate peak.

The isolated outlier should therefore no longer hijack the starting centre. Importantly, we still fit the **original data** — the outlier has not silently been removed.

In [ ]:
messy_result = fit_measurement(
    messy_measurement["x"],
    messy_measurement["y"],
    messy_measurement["yerr"],
)

print(messy_result.keys())

if messy_result["success"]:
    print("Valid points:", messy_result["n_valid"])
    print("Reduced chi2:", messy_result["reduced_chi2"])
    print(
        "Largest |standardized residual|:",
        messy_result["max_abs_standardized_residual"],
    )

    outlier_candidate = (
        messy_result["max_abs_standardized_residual"] > 8
    )
    print("Extreme outlier candidate:", outlier_candidate)

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.errorbar(
        messy_result["x"],
        messy_result["y"],
        yerr=messy_result["yerr"],
        fmt=".",
        ms=3,
    )
    ax.plot(messy_result["x"], messy_result["model"])
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 2.5))
    ax.axhline(0, lw=1)
    ax.plot(
        messy_result["x"],
        messy_result["standardized_residuals"],
        ".",
        ms=3,
    )
    ax.axhline(3, ls="--", lw=1)
    ax.axhline(-3, ls="--", lw=1)
    ax.set_xlabel("x")
    ax.set_ylabel("residual / uncertainty")
    plt.show()

The `NaN` values and invalid uncertainties can be rejected objectively because they are not valid measurements.

The large positive outlier is different. The pipeline now **flags** it as an extreme standardized residual without automatically deleting it:

```text
invalid datum     -> can be masked objectively
extreme residual  -> suspicious; investigate before masking
```

The outlier now damages the fit quality rather than preventing the optimiser from starting in the right place. That is much more useful behaviour for an automated pipeline.

## Solution 12 — Final pipeline

The final pipeline now treats model choice as part of the measurement process:

1. validate the data,
2. fit one Gaussian,
3. check whether the signal is significant,
4. accept a good single-Gaussian fit,
5. if its reduced $\chi^2$ is poor, try two Gaussians,
6. adopt two components only if the BIC improves substantially,
7. otherwise return a suspicious / bad fit rather than forcing an interpretation.

In [ ]:
def measure_signal(
    x,
    y,
    yerr,
    detection_threshold=3.0,
    chi2_trigger=1.8,
    delta_bic_required=10.0,
):
    single = fit_measurement(x, y, yerr)

    if not single.get("success", False):
        return single

    amplitude, centre, sigma, background = single["params"]
    amp_err, centre_err, sigma_err, background_err = single["errors"]

    amplitude_significance = (
        amplitude / amp_err
        if np.isfinite(amp_err) and amp_err > 0
        else np.nan
    )

    single.update({
        "amplitude": amplitude,
        "amplitude_error": amp_err,
        "centre": centre,
        "centre_error": centre_err,
        "sigma": sigma,
        "sigma_error": sigma_err,
        "background": background,
        "background_error": background_err,
        "amplitude_significance": amplitude_significance,
        "outlier_candidate": (
            single["max_abs_standardized_residual"] > 8
        ),
    })

    # No convincing signal.
    if (
        not np.isfinite(amplitude_significance)
        or amplitude_significance < detection_threshold
    ):
        single["status"] = "non_detection"
        return single

    # One Gaussian is adequate.
    if (
        0.3 < sigma < 30
        and single["reduced_chi2"] <= chi2_trigger
    ):
        single["status"] = "detection"
        return single

    # Significant signal, but poor single-Gaussian fit:
    # attempt a more complex model.
    double = fit_double_gaussian(single)

    if not double.get("success", False):
        single["status"] = "bad_fit"
        single["model_comparison"] = "double_fit_failed"
        return single

    delta_bic = single["bic"] - double["bic"]

    double["delta_bic_vs_single"] = delta_bic
    double["single_reduced_chi2"] = single["reduced_chi2"]
    double["single_bic"] = single["bic"]
    double["outlier_candidate"] = (
        double["max_abs_standardized_residual"] > 8
    )

    component_significances = [
        component["amplitude_significance"]
        for component in double["components"]
    ]

    double_is_justified = (
        delta_bic > delta_bic_required
        and min(component_significances) >= detection_threshold
        and double["reduced_chi2"] < single["reduced_chi2"]
    )

    if double_is_justified:
        double["status"] = "detection"
        return double

    # Poor fit remains unexplained by a justified second component.
    single["status"] = "bad_fit"
    single["model_comparison"] = "double_not_justified"
    single["delta_bic_vs_double"] = delta_bic

    return single

In [ ]:
final_results = {
    name: measure_signal(
        m["x"],
        m["y"],
        m["yerr"],
    )
    for name, m in measurements.items()
}

print(
    f"{'name':15s} "
    f"{'status':15s} "
    f"{'model':18s} "
    f"{'red.chi2':>10s} "
    f"{'Delta BIC':>10s}"
)

for name, r in final_results.items():
    print(
        f"{name:15s} "
        f"{r.get('status', 'unknown'):15s} "
        f"{r.get('model_type', 'unknown'):18s} "
        f"{r.get('reduced_chi2', np.nan):10.2f} "
        f"{r.get('delta_bic_vs_single', np.nan):10.2f}"
    )

print("\nDetailed adopted parameters:")

for name, r in final_results.items():
    print(f"\n{name}: {r['status']} / {r.get('model_type')}")

    if r.get("model_type") == "single_gaussian":
        print(
            f"  amplitude = {r.get('amplitude', np.nan):.3f} "
            f"+/- {r.get('amplitude_error', np.nan):.3f}"
        )
        print(f"  centre    = {r.get('centre', np.nan):.3f}")

    elif r.get("model_type") == "double_gaussian":
        for i, component in enumerate(r["components"], start=1):
            print(
                f"  component {i}: "
                f"A = {component['amplitude']:.3f} +/- "
                f"{component['amplitude_error']:.3f}, "
                f"centre = {component['centre']:.3f}, "
                f"sigma = {component['sigma']:.3f}"
            )

## Solution 13 — Reveal the hidden truth

Only now do we compare our conclusions with how the synthetic measurements were actually generated.

In [ ]:
truth_by_measurement = {
    f"measurement_{j}": _truth[i]
    for j, i in enumerate(order)
}

for name in measurements:
    print(name, "->", truth_by_measurement[name])

### What should you notice?

There are four fundamentally different situations:

1. **Noise only**  
   A fitting algorithm can still return a best-fitting Gaussian. The existence of fitted parameters does not establish that a signal is real.

2. **Strong single Gaussian**  
   This is the ideal development and validation case.

3. **Weak single Gaussian**  
   This tests whether your detection criteria behave sensibly near the boundary between measurement and non-detection.

4. **Two overlapping Gaussians**  
   A single-Gaussian fit may be highly significant but scientifically inadequate. The structured residuals are evidence that the model is missing information present in the data.

The last case is often the most important lesson:

> **A precise answer from the wrong model can be more dangerous than an obvious non-detection.**

# Optional extension — Write a deliberately imperfect FITS file

If `astropy` is available, you can package the measurements into a FITS table and deliberately include some invalid values.

This can be used as a follow-up exercise in which students must first understand the file structure and validate the inputs before applying their measurement pipeline.

In [ ]:
# OPTIONAL INSTRUCTOR CELL
#
# from astropy.io import fits
#
# names = list(measurements.keys())
#
# cols = [
#     fits.Column(
#         name="x",
#         format=f"{len(x)}D",
#         array=np.array([measurements[n]["x"] for n in names]),
#     ),
#     fits.Column(
#         name="flux",
#         format=f"{len(x)}D",
#         array=np.array([measurements[n]["y"] for n in names]),
#     ),
#     fits.Column(
#         name="flux_err",
#         format=f"{len(x)}D",
#         array=np.array([measurements[n]["yerr"] for n in names]),
#     ),
# ]
#
# hdu = fits.BinTableHDU.from_columns(cols)
# hdu.header["COMMENT"] = "Synthetic measurements for pipeline tutorial"
# hdu.writeto("mystery_measurements.fits", overwrite=True)

# Final takeaway

A measurement pipeline should not answer only:

> **What number best fits the data?**

It should also answer:

> **Is there evidence for a signal?**

> **Does the model describe the data adequately?**

> **Are the data themselves valid?**

> **How uncertain is the result?**

> **What should downstream users know before trusting it?**

That distinction — between *obtaining a fitted value* and *making a defensible measurement* — is central to scientific data analysis.